![Week 2 Day 2](../../assets/banners/w2d2.png)

# Week 2 Day 2 - SOLUTION - Chunking the Databases Unit

Reference solution for the exercise at the end of `week2/day2.ipynb`. Have a go yourself first: struggling a little is where the learning happens! Then compare.

**The exercise:** chunk `unit3_databases.pdf` at sizes 300, 1000 and 2000 with the recursive splitter and with `chunk_documents`, counting chunks and section-crossings; find the shortest structure-aware chunk; turn off the contextual header and find a chunk that no longer names its topic.

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK for most of this lab: the offline cells use fakes.")
        print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

print(settings.describe_models())
print(datasets.describe())

In [ ]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter

from gradewise.config import DATA_DIR
from gradewise.ingestion import chunk_documents, load_pdf

pages = load_pdf(DATA_DIR / "knowledge-base" / "unit3_databases.pdf")
full_text = "\n\n".join(p.page_content for p in pages)
section_starts = [m.start() for m in re.finditer(r"^## ", full_text, flags=re.MULTILINE)]
print(f"{len(pages)} pages, {len(full_text):,} characters, {len(section_starts)} sections")

## Part 1: Counting chunks and section-crossings

For the recursive splitter, a chunk "crosses" if a `## ` heading starts strictly inside it. For `chunk_documents`, the simplest test is the same idea applied to the chunk's own text: does a `## ` heading appear anywhere *after* its first line?

In [ ]:
def recursive_stats(size: int) -> tuple[int, int]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0, add_start_index=True)
    docs = splitter.create_documents([full_text])
    spans = [(d.metadata["start_index"], d.metadata["start_index"] + len(d.page_content)) for d in docs]
    crossing = sum(any(s + 5 < h < e - 5 for h in section_starts) for s, e in spans)
    return len(docs), crossing


def structured_stats(size: int) -> tuple[int, int]:
    chunks = chunk_documents(pages, chunk_size=size, chunk_overlap=0, contextual_header=False)
    crossing = sum("\n## " in c.page_content.strip().split("\n", 1)[-1] for c in chunks)
    return len(chunks), crossing


print(f"{'size':>6} | {'recursive: chunks, crossing':>28} | {'chunk_documents: chunks, crossing':>34}")
for size in (300, 1000, 2000):
    (rn, rc), (sn, sc) = recursive_stats(size), structured_stats(size)
    print(f"{size:>6} | {rn:>16}, {rc:>9} | {sn:>22}, {sc:>9}")

The recursive splitter crosses more section boundaries as chunks grow (a 2,000-character window almost always straddles two sections). `chunk_documents` never does, at any size, because it splits on headings *before* it measures length.

## Part 2: The shortest chunk

In [ ]:
chunks = chunk_documents(pages)
shortest = min(chunks, key=lambda c: len(c.page_content))
meta = shortest.metadata
print(f"{len(shortest.page_content)} characters, section {meta['section']!r}, page {meta['page']}:\n")
print(shortest.page_content)

The shortest chunk is the unit's title block: a genuinely short "section" with no neighbour to merge into, so `chunk_documents` keeps it as it is. 🔁 It used to be worse: an early version of the chunker left a **one-word** chunk here (just `storage.`, the leftover tail of the Unit summary). The fix is the `min_chunk_size=200` parameter: chunks shorter than that are merged into a neighbour from the **same section**. Try `chunk_documents(pages, min_chunk_size=0)` to bring the leftovers back.

## Part 3: Without the contextual header

In [ ]:
plain = chunk_documents(pages, contextual_header=False)
for chunk in plain:
    topic_words = re.findall(r"[a-z]{4,}", chunk.metadata["section"].lower())
    if not any(w in chunk.page_content.lower() for w in topic_words):
        print(f"§{chunk.metadata['section']} (p{chunk.metadata['page']}) never mentions {topic_words}:\n")
        print(chunk.page_content[:400], "…")
        break
else:
    print("Every chunk happens to mention its section's words.")

That chunk is about its section, but nothing in its text says so. With `contextual_header=True` it would start with `[CS201 Unit 3: Database Systems > <section>]`, so both the embedding and BM25 would know its topic.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">All of this lives in <code>gradewise/ingestion/chunking.py</code>: the header-first split, the recursive fallback and the contextual header.</span>
        </td>
    </tr>
</table>